# CONSULTAS SOBRE LOS DATOS GENERADOS  

## Consultas sobre los datos generados

Se plantean 5 consultas analíticas sobre datos cruzados entre tablas. Cada una limita su salida a 15 filas (`LIMIT 15`).

Se ejecutan en el **dialecto SQL heredado (legacy)**: `QueryJobConfig(use_legacy_sql=True)`.

| Query | Cruce de tablas | Qué mide |
|---|---|---|
| `q1` | `customers` × `orders` × `order_items` | Los 15 clientes con mayor volumen de negocio (GMV) |
| `q2` | `order_items` × `products` × `categories` | Top 15 productos por ingresos y margen bruto |
| `q3` | `reviews` × `products` × `categories` | Rating medio y nº de reseñas por categoría |
| `q4` | `orders` × `customers` × `order_items` | GMV mensual por país (top 15 combinaciones país-mes) |
| `q5` | `orders` (auto-join) × `customers` | Repetición de compra en ≤ 30 días |


In [1]:
###########################################
####### EJECUTA PARA CONEXIÓN  ############
###########################################

from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv
load_dotenv()

# Ruta absoluta al proyecto
carpeta = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
# Configuración
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

CREDENTIALS_PATH = os.path.abspath(
    os.path.join(carpeta, os.pardir, "credentials", "hglebredo-online-92ccd6898cd9.json")
)

if not os.path.isfile(CREDENTIALS_PATH):
    raise FileNotFoundError(f"Credentials file not found: {CREDENTIALS_PATH}")

credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)


In [42]:
q1 = f"""
// QUERY 1: Los 15 clientes con mayor volumen de negocio
// Cruce: customers -> orders -> order_items
SELECT
  c.customer_id AS customer_id,
  c.first_name + ' ' + c.last_name AS cliente,
  c.country AS country,
  COUNT(DISTINCT o.order_id) AS num_pedidos,
  ROUND(SUM(CAST(oi.unit_price AS FLOAT) * oi.quantity), 2) AS total_gastado
FROM [{PROJECT_ID}:{DATASET_ID}.customers] c

JOIN [{PROJECT_ID}:{DATASET_ID}.orders] o
  ON o.customer_id = c.customer_id

JOIN [{PROJECT_ID}:{DATASET_ID}.order_items] oi
  ON oi.order_id = o.order_id
GROUP BY customer_id, cliente, country
ORDER BY total_gastado DESC
LIMIT 15
"""

q2 = f"""
// QUERY 2: Top 15 productos por ingresos y margen bruto
// Cruce: order_items -> products -> categories
SELECT
  c.name AS categoria,
  p.name AS producto,
  p.product_id AS product_id,
  SUM(oi.quantity) AS unidades_vendidas,
  ROUND(SUM(CAST(oi.unit_price AS FLOAT) * oi.quantity), 2) AS ingresos,
  ROUND(SUM(CAST(oi.unit_price AS FLOAT) * oi.quantity - CAST(p.cost AS FLOAT) * oi.quantity), 2) AS margen
FROM [{PROJECT_ID}:{DATASET_ID}.order_items] oi

JOIN [{PROJECT_ID}:{DATASET_ID}.products] p
  ON p.product_id = oi.product_id

JOIN [{PROJECT_ID}:{DATASET_ID}.categories] c
  ON c.category_id = p.category_id
GROUP BY categoria, producto, product_id
ORDER BY ingresos DESC
LIMIT 15
"""

q3 = f"""
// QUERY 3: Calidad por categoria: rating medio y numero de resenas
// Cruce: reviews -> products -> categories
SELECT
  c.name AS categoria,
  COUNT(r.review_id) AS num_resenas,
  ROUND(AVG(r.rating), 2) AS rating_medio
FROM [{PROJECT_ID}:{DATASET_ID}.reviews] r

JOIN [{PROJECT_ID}:{DATASET_ID}.products] p
  ON p.product_id = r.product_id

JOIN [{PROJECT_ID}:{DATASET_ID}.categories] c
  ON c.category_id = p.category_id
GROUP BY categoria
ORDER BY rating_medio DESC, num_resenas DESC
LIMIT 15
"""

q4 = f"""
// QUERY 4: GMV mensual por paises: top 15 combinaciones pais-mes
// Cruce: orders -> customers -> order_items
SELECT
  c.country AS pais,
  SUBSTR(CAST(o.order_date AS STRING), 1, 7) AS mes,
  COUNT(DISTINCT o.order_id) AS num_pedidos,
  ROUND(SUM(CAST(oi.unit_price AS FLOAT) * oi.quantity), 2) AS total_gastado
FROM [{PROJECT_ID}:{DATASET_ID}.orders] o

JOIN [{PROJECT_ID}:{DATASET_ID}.customers] c
  ON c.customer_id = o.customer_id

JOIN [{PROJECT_ID}:{DATASET_ID}.order_items] oi
  ON oi.order_id = o.order_id
GROUP BY pais, mes
ORDER BY total_gastado DESC
LIMIT 15
"""

q5 = f"""
// QUERY 5: Repeticion de compra en <= 30 dias
// Cruce: orders (auto-join) -> customers
SELECT
  c.first_name + ' ' + c.last_name AS cliente,
  o2.customer_id AS customer_id,
  o2.order_id AS pedido,
  o1.order_id AS pedido_anterior,
  DATEDIFF(o2.order_date, o1.order_date) AS dias_desde_anterior
FROM [{PROJECT_ID}:{DATASET_ID}.orders] o2

JOIN [{PROJECT_ID}:{DATASET_ID}.orders] o1
  ON o1.customer_id = o2.customer_id

JOIN [{PROJECT_ID}:{DATASET_ID}.customers] c
  ON c.customer_id = o2.customer_id
WHERE o1.order_date < o2.order_date
  AND DATEDIFF(o2.order_date, o1.order_date) <= 30
ORDER BY dias_desde_anterior
LIMIT 15
"""


In [49]:
###########################################
####### EJECUTA PARA LANZAR QUERYS ########
###########################################

# Dialecto SQL heredado (legacy)
job_config = bigquery.QueryJobConfig(use_legacy_sql=True)

for i in range(1, 5):
    print(f"\n--- Query {i} ------------>")
    
    query_sql = globals()[f"q{i}"]
    resultado = client.query_and_wait(
        query_sql, job_config=job_config)
    for row in resultado:
        print(row)



--- Query 1 ------------>
Row((365, 'Jordan Hudson', 'Belgium', 10, 55413.8), {'customer_id': 0, 'cliente': 1, 'country': 2, 'num_pedidos': 3, 'total_gastado': 4})
Row((96, 'Elizabeth Marshall', 'Slovakia', 9, 55343.52), {'customer_id': 0, 'cliente': 1, 'country': 2, 'num_pedidos': 3, 'total_gastado': 4})
Row((188, 'Luke Miller', 'Germany', 10, 51381.31), {'customer_id': 0, 'cliente': 1, 'country': 2, 'num_pedidos': 3, 'total_gastado': 4})
Row((267, 'Gabriella Ryan', 'Malta', 6, 50211.53), {'customer_id': 0, 'cliente': 1, 'country': 2, 'num_pedidos': 3, 'total_gastado': 4})
Row((41, 'Tyrone Moran', 'United Kingdom', 7, 46510.57), {'customer_id': 0, 'cliente': 1, 'country': 2, 'num_pedidos': 3, 'total_gastado': 4})
Row((258, 'Crystal Vaughn', 'Luxembourg', 8, 43524.41), {'customer_id': 0, 'cliente': 1, 'country': 2, 'num_pedidos': 3, 'total_gastado': 4})
Row((163, 'Carolyn Smith', 'Ireland', 6, 41534.82), {'customer_id': 0, 'cliente': 1, 'country': 2, 'num_pedidos': 3, 'total_gastado':